In [1]:
"""Clinical (% ED visits) vs. wastewater qPCR (copies/mL, 4-week rolling mean)
time series, one chart per virus: SARS-CoV-2, Flu (A+B combined), RSV (A+B
combined).

This is the qPCR-vs-clinical counterpart of figure4A.ipynb (sequencing rpkmf
vs. clinical) -- same formatting (colors, correlation-box style/position,
legend, title convention, axis label styling), applied to the qPCR side of
the pipeline instead of the sequencing side.

Missing weeks (no qPCR sample that week, or -- for the clinical side -- any
gap in the ED file) are bridged with a dashed segment rather than a solid
line or a blank gap.

"""

#Import required packages
import math
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.ticker import FixedLocator, FuncFormatter
from scipy.stats import pearsonr, spearmanr

#Input file paths
DATA_PATH = "Data_4b_Clinical_vs_qPCR_Data_Final.xlsx"

COLOR_WW = "#3D2412"   # dataviz categorical slot 1 (brown) -- wastewater (qPCR copies/mL)
COLOR_ED = "#2a78d6"   # dataviz categorical slot 2 (blue) -- clinical (% ED visits)

GAP_LINESTYLE = (0, (4, 3))  # dashed, used by plot_gap_aware_series for missing-period bridges
FLOOR_LINESTYLE = (0, (1, 2))  # dots, used for segments sitting flat at the series' zero-substituted floor

VIRUSES = [
    {
        "sheet": "SARS-COV-2",
        "perc_col": "PERC_COVID",
        "value_col": "covid_copiesperml_rolling_4wk",
        "title": "SARS-CoV-2",
        "out_path": "fig4g_i_sars_cov2_qpcr_vs_clinical_rolling.png",
    },
    {
        "sheet": "FLU",
        "perc_col": "PERC_INFLUENZA",
        "value_col": "flu_combined_copiesperml_rolling_4wk",
        "title": "Influenza (A+B combined)",
        "out_path": "fig4g_i_flu_qpcr_vs_clinical_rolling.png",
        "box_side": "left",
    },
    {
        "sheet": "RSV",
        "perc_col": "PERC_RSV",
        "value_col": "rsv_combined_copiesperml_rolling_4wk",
        "title": "RSV (A+B combined)",
        "out_path": "fig4g_i_rsv_qpcr_vs_clinical_rolling.png",
        "box_side": "left",
    },
]


def compute_correlations(sub, perc_col, value_col):
    """Pearson and Spearman correlation between the two series, restricted to
    weeks where both are present (pandas .corr() would otherwise silently
    pairwise-drop, but we want n reported explicitly).
    """
    paired = sub[[perc_col, value_col]].dropna()
    n = len(paired)
    if n < 2:
        return None, None, None, None, n
    pearson_r, pearson_p = pearsonr(paired[perc_col], paired[value_col])
    spearman_r, spearman_p = spearmanr(paired[perc_col], paired[value_col])
    return pearson_r, pearson_p, spearman_r, spearman_p, n


def plot_gap_aware_series(ax, x, y, color, label=None, linewidth=3.25, marker=None, markersize=None, zorder=3, floor_mask=None):
    """Plot a period-indexed series (x = one row per period on a regular
    weekly/monthly grid, already sorted; y may be NaN for periods with no
    data -- e.g. the result of merging onto the ED file's full weekly/monthly
    index) as solid segments between temporally-adjacent known points, and a
    dashed segment (GAP_LINESTYLE) bridging directly across any run of one or
    more missing periods. This makes a real gap in the underlying data read
    as "unknown" rather than a plain connected line (implies observed data
    across the gap) or a blank stretch (easy to misread as zero).

    x/y must be positional-index-aligned with the full period grid so that a
    gap in the data is exactly a gap in array position, not something inferred
    from timestamp arithmetic.

    floor_mask, if given, is a boolean sequence aligned with y marking points
    whose real value was an actual zero (displayed at the series' nonzero
    floor instead of true 0 -- see data_floor substitution in make_chart). A
    segment where BOTH endpoints are floor points is drawn with FLOOR_LINESTYLE
    (dotted) instead of solid, since it represents a "nothing detected" flat
    run rather than an observed trend sitting at that level. A segment
    touching a floor point on only one end (a real rise out of, or fall into,
    the floor) stays solid.
    """
    x = list(x)
    y = list(y)
    floor_mask = list(floor_mask) if floor_mask is not None else [False] * len(y)
    known = [i for i, v in enumerate(y) if pd.notna(v)]

    if label is not None:
        ax.plot([], [], color=color, linewidth=linewidth, marker=marker,
                 markersize=markersize, linestyle="-", label=label, zorder=zorder)

    # Group consecutive same-style segments into ONE polyline per run rather
    # than one plot() call per week-to-week pair -- a dash/dot pattern only
    # cycles correctly across a continuous artist. A multi-week flat floor
    # run drawn as many separate 1-week-long segments each restarts the dash
    # phase from 0, and with round end-caps on a segment barely longer than
    # one dash period, each mini-segment renders almost fully "on" -- the
    # whole run then looks solid instead of dotted.
    runs = []  # (style, [point indices])
    for a, b in zip(known[:-1], known[1:]):
        if b - a != 1:
            style = GAP_LINESTYLE
        elif floor_mask[a] and floor_mask[b]:
            style = FLOOR_LINESTYLE
        else:
            style = "-"
        if runs and runs[-1][0] == style and runs[-1][1][-1] == a:
            runs[-1][1].append(b)
        else:
            runs.append((style, [a, b]))

    for style, idxs in runs:
        ax.plot([x[i] for i in idxs], [y[i] for i in idxs], color=color, linewidth=linewidth,
                 marker=marker, markersize=markersize, linestyle=style, zorder=zorder,
                 dash_capstyle="round" if style == FLOOR_LINESTYLE else "butt")


def _style_axes(ax_left, ax_right):
    ax_left.set_ylabel("Wastewater qPCR \n (copies/mL, 4-week rolling mean)", fontsize=18, color="black", fontweight="bold")
    ax_left.tick_params(axis="y", colors="black", labelsize=18)
    ax_left.spines["left"].set_color("black")

    ax_right.set_ylabel("% ED visits (Statewide)", fontsize=18, color="black", fontweight="bold")
    ax_right.tick_params(axis="y", colors="black",labelsize=18)
    ax_right.spines["right"].set_color("black")
    ax_right.spines["left"].set_visible(False)

    ax_left.spines["top"].set_visible(False)
    ax_right.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)

    ax_left.set_ylim(bottom=0)
    ax_right.set_ylim(bottom=0)

    ax_left.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_left.set_axisbelow(True)


def _format_p(p):
    return f"p = {p:.2e}"


def _configure_symlog_axis(ax, values, top, mark_floor=True):
    """Symlog y-scale sized to `values`' own floor (nonzero min) and `top`.
    When `mark_floor` is True, 0 and the floor each get an explicit labeled
    tick; when False, the floor is only used to size the symlog scale itself
    (so the low end doesn't collapse into the near-zero linear band) without
    surfacing it as its own tick/label -- used for the % ED axis, which has
    no real "limit of detection" concept, unlike qPCR copies/mL.

    Used for BOTH axes here (wastewater qPCR copies/mL AND % ED visits), not
    just the left one -- the two series' underlying values differ by many
    orders of magnitude in absolute units, but their proportional
    (order-of-magnitude) swings from trough to peak are actually similar
    (qPCR spans ~2.7 decades, ED ~1.9 decades). Pairing one log-scaled axis
    with a linear one (the original design) keeps the linear series pinned
    near the bottom at its own troughs while the log series stays elevated,
    so the two lines occupy different vertical bands even when they track
    each other closely. Putting both on a matched log-shaped scale lets
    proportionally-similar swings render at proportionally-similar heights
    on both axes, which is what actually makes them visually overlap --
    matching what happens with the sequencing-vs-clinical chart, where both
    series' natural ranges happen to be small enough that they're
    effectively both linear already.
    """
    nonzero = values[values > 0]
    floor = nonzero.min() if len(nonzero) else 1
    linthresh = floor if floor > 0 else 1
    ax.set_yscale("symlog", linthresh=linthresh)
    ax.set_ylim(0, top)

    # Only keep default log-decade ticks that fall ABOVE the floor -- ticks
    # matplotlib generates between 0 and the floor sit inside the linear
    # (sub-linthresh) stretch of the axis and just crowd up against the
    # explicit 0/floor labels instead of marking a meaningful log decade.
    default_ticks = [t for t in ax.get_yticks() if floor < t <= top]
    all_ticks = sorted(set(default_ticks) | ({0, floor} if mark_floor else {0}))
    ax.yaxis.set_major_locator(FixedLocator(all_ticks))

    def _format_ytick(y, pos):
        if y == 0:
            return "0"
        if mark_floor and math.isclose(y, floor, rel_tol=1e-9):
            return f"{floor:.3g}"
        exp = math.log10(y)
        if math.isclose(exp, round(exp), abs_tol=1e-6):
            return f"$10^{{{int(round(exp))}}}$"
        return ""

    ax.yaxis.set_major_formatter(FuncFormatter(_format_ytick))

    if mark_floor:
        for tick_val, tick_label in zip(all_ticks, ax.get_yticklabels()):
            if math.isclose(tick_val, floor, rel_tol=1e-9):
                tick_label.set_fontweight("bold")

    return floor


def make_chart(virus, df):
    sub = df[["WEEK_ENDING", virus["perc_col"], virus["value_col"]]].sort_values("WEEK_ENDING").reset_index(drop=True)

    n_weeks = len(sub)
    n_with_qpcr = sub[virus["value_col"]].notna().sum()
    print(f"{virus['title']}: {n_weeks} weeks total, {n_with_qpcr} with qPCR data -> {virus['out_path']}")

    pearson_r, pearson_p, spearman_r, spearman_p, n_paired = compute_correlations(sub, virus["perc_col"], virus["value_col"])
    #print(f"{virus['title']}: Pearson r={pearson_r:.3f} (p={pearson_p:.2e}), ",
    #      f"Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={n_paired} weeks)")
    print(f"{virus['title']}: Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={n_paired} weeks)")
    
    nonzero = sub[virus["value_col"]][sub[virus["value_col"]] > 0]
    data_floor = nonzero.min() if len(nonzero) else 0

    # Plot actual zero-value weeks (rolling mean is exactly 0 when nothing in
    # the trailing 4-week window was positive) AT the series' nonzero floor
    # instead of true 0 -- otherwise those weeks still visually crash the
    # line to the bottom of the axis even with 0 shown as the axis limit.
    # Correlations above are computed on the unmodified `sub` (real zeros),
    # only the plotted line is substituted.
    plot_values = sub[virus["value_col"]].replace(0, data_floor)
    was_zero = sub[virus["value_col"]] == 0

    fig, ax_left = plt.subplots(figsize=(15, 7))
    ax_right = ax_left.twinx()
    
    plot_gap_aware_series(ax_left, sub["WEEK_ENDING"], plot_values, COLOR_WW,
                           label="Wastewater qPCR (copies/mL, 4-week rolling mean)",
                           floor_mask=was_zero)
    plot_gap_aware_series(ax_right, sub["WEEK_ENDING"], sub[virus["perc_col"]], COLOR_ED,
                           label="% ED visits (Statewide)")

    _style_axes(ax_left, ax_right)

    _configure_symlog_axis(ax_left, plot_values, top=plot_values.max() * 1.6)
    _configure_symlog_axis(ax_right, sub[virus["perc_col"]], top=sub[virus["perc_col"]].max() * 1.6, mark_floor=False)

    ax_left.xaxis.set_major_locator(mdates.YearLocator())
    ax_left.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax_left.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_left.set_xlabel("Date (Week ending)", fontsize=18, fontweight="bold")
    ax_left.tick_params(axis="y", labelsize=18)
    ax_left.tick_params(axis="x", labelsize=18)
    box_side = virus.get("box_side", "right")
    box_x, box_ha = (0.015, "left") if box_side == "left" else (0.985, "right")
    ax_left.text(
        box_x, 0.97,
        #f"Pearson r = {pearson_r:.2f} ({_format_p(pearson_p)})\n"
        f"Spearman ρ = {spearman_r:.2f} ({_format_p(spearman_p)})\n"
        f"n = {n_paired} weeks",
        transform=ax_left.transAxes, fontsize=18, va="top", ha=box_ha,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9),
        zorder=5)

    lines_left, labels_left = ax_left.get_legend_handles_labels()
    lines_right, labels_right = ax_right.get_legend_handles_labels()
    ax_left.legend(lines_left + lines_right, labels_left + labels_right,
                   loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False, fontsize=18)
    ax_left.set_title(
        f"{virus['title']}: Wastewater qPCR Signal vs. ED Visit Rate, Weekly Rolling Average",
        fontsize=18, fontweight="bold", loc="center", pad=20,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 1])

    out_dir = os.path.dirname(virus["out_path"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path"], dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(virus["out_path"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {virus['out_path']}")


def main():
    xl = pd.ExcelFile(DATA_PATH)
    for virus in VIRUSES:
        df = xl.parse(virus["sheet"])
        make_chart(virus, df)


if __name__ == "__main__":
    main()


SARS-CoV-2: 234 weeks total, 190 with qPCR data -> fig4g_i_sars_cov2_qpcr_vs_clinical_rolling.png
SARS-CoV-2: Spearman rho=0.800 (p=1.40e-43) (n=190 weeks)
Saved fig4g_i_sars_cov2_qpcr_vs_clinical_rolling.png
Influenza (A+B combined): 234 weeks total, 190 with qPCR data -> fig4g_i_flu_qpcr_vs_clinical_rolling.png
Influenza (A+B combined): Spearman rho=0.581 (p=1.54e-18) (n=190 weeks)
Saved fig4g_i_flu_qpcr_vs_clinical_rolling.png
RSV (A+B combined): 234 weeks total, 190 with qPCR data -> fig4g_i_rsv_qpcr_vs_clinical_rolling.png
RSV (A+B combined): Spearman rho=0.288 (p=5.64e-05) (n=190 weeks)
Saved fig4g_i_rsv_qpcr_vs_clinical_rolling.png
